# Migrate from A/B Testing to Bandits

**Goal:** Learn how to safely transition from traditional A/B tests to adaptive bandit algorithms without disrupting operations.

**What you'll learn:**
- Compare pure A/B, pure bandit, and hybrid approaches on same problem
- Implement a hybrid strategy with burn-in period
- Migration checklist for production systems

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

In [ ]:
learning_objectives(["**Start with shadow mode** — Run bandit offline first to validate", "**Use burn-in period** — Initial A/B test provides reliable starting point", "**Keep baseline running** — 10% A/B traffic for comparison", "**Log propensities** — Enable offline evaluation of new policies", "**Monitor intensively** — First 2-4 weeks require close attention"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Step 1: Simulate Existing A/B Test

Traditional 50/50 split test comparing two commodity allocation strategies.

In [ ]:
class ABTest:
    """Traditional A/B test with fixed allocation."""

    def __init__(self, variants):
        self.variants = variants  # List of variant names
        self.n_variants = len(variants)
        self.counts = defaultdict(int)
        self.rewards = defaultdict(list)

    def select_variant(self):
        """Uniform random selection (equal allocation)."""
        variant = np.random.choice(self.variants)
        self.counts[variant] += 1
        return variant

    def record_reward(self, variant, reward):
        self.rewards[variant].append(reward)

    def get_stats(self):
        return {
            variant: {
                "mean": np.mean(self.rewards[variant]) if self.rewards[variant] else 0,
                "std": np.std(self.rewards[variant]) if self.rewards[variant] else 0,
                "n": self.counts[variant]
            }
            for variant in self.variants
        }


# True conversion rates (unknown to the test)
true_means = {
    "Strategy_A": 0.55,  # Better strategy
    "Strategy_B": 0.45   # Worse strategy
}

# Run A/B test for 1000 trials
ab_test = ABTest(list(true_means.keys()))
ab_rewards = []
ab_regrets = []

best_mean = max(true_means.values())

for t in range(1000):
    variant = ab_test.select_variant()
    reward = 1.0 if np.random.random() < true_means[variant] else 0.0
    ab_test.record_reward(variant, reward)

    ab_rewards.append(reward)
    ab_regrets.append(best_mean - true_means[variant])

# Results
print("A/B TEST RESULTS (1000 trials)")
print("=" * 50)
for variant, stats in ab_test.get_stats().items():
    print(f"{variant}:")
    print(f"  Trials: {stats['n']}")
    print(f"  Mean: {stats['mean']:.3f} ± {stats['std']:.3f}")
    print(f"  True mean: {true_means[variant]:.3f}")

print(f"\nCumulative regret: {sum(ab_regrets):.2f}")
print(f"Average reward: {np.mean(ab_rewards):.3f}")

## Step 2: Compare with Pure Bandit

Run Thompson Sampling on the same problem.

In [ ]:
class ThompsonSampling:
    """Thompson Sampling bandit."""

    def __init__(self, arms):
        self.arms = arms
        self.alpha = {arm: 1.0 for arm in arms}  # Beta prior
        self.beta = {arm: 1.0 for arm in arms}
        self.counts = {arm: 0 for arm in arms}

    def select_arm(self):
        """Thompson sampling: sample from Beta posterior."""
        samples = {arm: np.random.beta(self.alpha[arm], self.beta[arm])
                  for arm in self.arms}
        arm = max(samples, key=samples.get)
        self.counts[arm] += 1
        return arm

    def update(self, arm, reward):
        """Update Beta posterior."""
        self.alpha[arm] += reward
        self.beta[arm] += (1 - reward)

    def get_stats(self):
        return {
            arm: {
                "mean": self.alpha[arm] / (self.alpha[arm] + self.beta[arm]),
                "n": self.counts[arm]
            }
            for arm in self.arms
        }


# Run Thompson Sampling for 1000 trials
bandit = ThompsonSampling(list(true_means.keys()))
bandit_rewards = []
bandit_regrets = []

for t in range(1000):
    arm = bandit.select_arm()
    reward = 1.0 if np.random.random() < true_means[arm] else 0.0
    bandit.update(arm, reward)

    bandit_rewards.append(reward)
    bandit_regrets.append(best_mean - true_means[arm])

# Results
print("\nTHOMPSON SAMPLING RESULTS (1000 trials)")
print("=" * 50)
for arm, stats in bandit.get_stats().items():
    print(f"{arm}:")
    print(f"  Trials: {stats['n']}")
    print(f"  Estimated mean: {stats['mean']:.3f}")
    print(f"  True mean: {true_means[arm]:.3f}")

print(f"\nCumulative regret: {sum(bandit_regrets):.2f}")
print(f"Average reward: {np.mean(bandit_rewards):.3f}")

## Step 3: Hybrid Approach (Burn-In Period)

Start as A/B test, switch to bandit after collecting initial data.

In [ ]:
class HybridABBandit:
    """Hybrid: A/B test during burn-in, then switch to bandit."""

    def __init__(self, arms, burnin_rounds=200):
        self.arms = arms
        self.burnin_rounds = burnin_rounds
        self.round = 0

        # A/B test phase
        self.ab_test = ABTest(arms)

        # Bandit phase
        self.bandit = ThompsonSampling(arms)

        self.mode = "ab_test"

    def select_arm(self):
        self.round += 1

        if self.round <= self.burnin_rounds:
            # A/B test mode
            self.mode = "ab_test"
            return self.ab_test.select_variant()
        else:
            # Bandit mode
            if self.mode == "ab_test":
                # First bandit round: initialize with A/B test data
                self._transfer_data()
                self.mode = "bandit"

            return self.bandit.select_arm()

    def update(self, arm, reward):
        if self.mode == "ab_test":
            self.ab_test.record_reward(arm, reward)
        else:
            self.bandit.update(arm, reward)

    def _transfer_data(self):
        """Transfer A/B test data to initialize bandit priors."""
        print(f"\n>>> Switching from A/B test to bandit at round {self.round}")
        for arm in self.arms:
            rewards = self.ab_test.rewards[arm]
            if rewards:
                successes = sum(rewards)
                failures = len(rewards) - successes
                self.bandit.alpha[arm] = 1.0 + successes
                self.bandit.beta[arm] = 1.0 + failures
        print(f"    Initialized bandit with {len(rewards)} samples per arm\n")

    def get_stats(self):
        if self.mode == "ab_test":
            return {"mode": "ab_test", "data": self.ab_test.get_stats()}
        else:
            return {"mode": "bandit", "data": self.bandit.get_stats()}


# Run hybrid approach
hybrid = HybridABBandit(list(true_means.keys()), burnin_rounds=200)
hybrid_rewards = []
hybrid_regrets = []

for t in range(1000):
    arm = hybrid.select_arm()
    reward = 1.0 if np.random.random() < true_means[arm] else 0.0
    hybrid.update(arm, reward)

    hybrid_rewards.append(reward)
    hybrid_regrets.append(best_mean - true_means[arm])

# Results
print("\nHYBRID APPROACH RESULTS (1000 trials)")
print("=" * 50)
stats = hybrid.get_stats()
print(f"Final mode: {stats['mode']}")
for arm, arm_stats in stats['data'].items():
    print(f"{arm}:")
    print(f"  Trials: {arm_stats['n']}")
    print(f"  Estimated mean: {arm_stats['mean']:.3f}")
    print(f"  True mean: {true_means[arm]:.3f}")

print(f"\nCumulative regret: {sum(hybrid_regrets):.2f}")
print(f"Average reward: {np.mean(hybrid_rewards):.3f}")

## Step 4: Compare All Three Approaches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative regret comparison
axes[0].plot(np.cumsum(ab_regrets), label='A/B Test', linewidth=2, alpha=0.8)
axes[0].plot(np.cumsum(bandit_regrets), label='Pure Bandit', linewidth=2, alpha=0.8)
axes[0].plot(np.cumsum(hybrid_regrets), label='Hybrid (burn-in=200)', linewidth=2, alpha=0.8)
axes[0].axvline(200, color='gray', linestyle='--', alpha=0.5, label='Burn-in ends')
axes[0].set_title('Cumulative Regret Comparison', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Cumulative Regret')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Average reward comparison (moving average)
window = 50
ab_ma = np.convolve(ab_rewards, np.ones(window)/window, mode='valid')
bandit_ma = np.convolve(bandit_rewards, np.ones(window)/window, mode='valid')
hybrid_ma = np.convolve(hybrid_rewards, np.ones(window)/window, mode='valid')

axes[1].plot(ab_ma, label='A/B Test', linewidth=2, alpha=0.8)
axes[1].plot(bandit_ma, label='Pure Bandit', linewidth=2, alpha=0.8)
axes[1].plot(hybrid_ma, label='Hybrid', linewidth=2, alpha=0.8)
axes[1].axhline(best_mean, color='green', linestyle='--', alpha=0.7, label='Best arm')
axes[1].axvline(200, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title(f'Reward Moving Average (window={window})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Average Reward')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Approach':<20} {'Total Regret':<15} {'Avg Reward':<15} {'Winner'}")
print("-" * 60)
print(f"{'A/B Test':<20} {sum(ab_regrets):<15.2f} {np.mean(ab_rewards):<15.3f}")
print(f"{'Pure Bandit':<20} {sum(bandit_regrets):<15.2f} {np.mean(bandit_rewards):<15.3f}")
print(f"{'Hybrid':<20} {sum(hybrid_regrets):<15.2f} {np.mean(hybrid_rewards):<15.3f} ✓")
print("\nHybrid approach balances exploration (A/B) with exploitation (bandit)!")

## Step 5: Migration Checklist

What to keep from A/B testing and what to add for bandits.

In [ ]:
print("A/B TO BANDIT MIGRATION CHECKLIST")
print("=" * 70)

print("\n✅ KEEP FROM A/B TESTING:")
checklist_keep = [
    "Statistical rigor: proper sample sizes, significance testing",
    "Logging infrastructure: decision tracking, outcome measurement",
    "Guardrails: business rules, position limits, safety checks",
    "Monitoring: performance dashboards, alert systems",
    "Documentation: clear experiment goals, success criteria"
]
for item in checklist_keep:
    print(f"   • {item}")

print("\n➕ ADD FOR BANDITS:")
checklist_add = [
    "Adaptive allocation: adjust traffic based on performance",
    "Exploration strategy: epsilon-greedy, Thompson Sampling, UCB",
    "Burn-in period: initial exploration phase with forced randomization",
    "Policy versioning: track which algorithm version made each decision",
    "Propensity logging: store P(arm|context) for offline evaluation",
    "Regret monitoring: track opportunity cost of suboptimal choices",
    "Cold start handling: strategy for new arms with no data",
    "Non-stationarity detection: monitor for distribution shifts"
]
for item in checklist_add:
    print(f"   • {item}")

print("\n⚠️  CHALLENGES TO EXPECT:")
challenges = [
    "Explaining adaptive allocation to stakeholders (not 50/50 anymore)",
    "Determining when exploration has 'converged' (bandits never fully stop exploring)",
    "Handling policy changes mid-experiment (version control is critical)",
    "Computing statistical significance (use offline evaluation methods)",
    "Debugging performance issues (comprehensive logging is essential)"
]
for item in challenges:
    print(f"   • {item}")

print("\n📋 RECOMMENDED MIGRATION PHASES:")
phases = [
    ("Week 1-2", "Shadow mode: Run bandit offline, keep A/B test live"),
    ("Week 3-4", "Parallel: 50% A/B test, 50% bandit with forced exploration"),
    ("Week 5-6", "Hybrid: Burn-in with A/B, then switch to bandit"),
    ("Week 7-8", "Bandit primary: 90% bandit, 10% A/B for baseline comparison"),
    ("Week 9+", "Full bandit: 100% adaptive allocation, continuous monitoring")
]
for week, phase in phases:
    print(f"   {week:<10} {phase}")

print("\n" + "=" * 70)

## Step 6: Test Different Burn-In Periods

In [ ]:
print("Testing different burn-in periods...\n")

burnin_periods = [0, 50, 100, 200, 400]
results = {}

for burnin in burnin_periods:
    if burnin == 0:
        # Pure bandit (no burn-in)
        b = ThompsonSampling(list(true_means.keys()))
        regrets = []
        for t in range(1000):
            arm = b.select_arm()
            reward = 1.0 if np.random.random() < true_means[arm] else 0.0
            b.update(arm, reward)
            regrets.append(best_mean - true_means[arm])
    else:
        # Hybrid with burn-in
        h = HybridABBandit(list(true_means.keys()), burnin_rounds=burnin)
        regrets = []
        for t in range(1000):
            arm = h.select_arm()
            reward = 1.0 if np.random.random() < true_means[arm] else 0.0
            h.update(arm, reward)
            regrets.append(best_mean - true_means[arm])

    results[burnin] = sum(regrets)

# Display results
print("Burn-in Period vs Total Regret:")
print("-" * 40)
for burnin, regret in sorted(results.items()):
    label = "Pure bandit" if burnin == 0 else f"Hybrid (burn-in={burnin})"
    print(f"{label:<25} {regret:>8.2f}")

best_burnin = min(results, key=results.get)
print(f"\n✓ Optimal burn-in period: {best_burnin} rounds")

## Modify This: Experiment with Migration Strategies

Try these modifications:
1. **Three-arm problem** — Add a third strategy, see how migration performs
2. **Non-stationary environment** — Change true means after 500 rounds, test adaptation
3. **Different priors** — Initialize bandit with optimistic vs pessimistic priors
4. **Staged rollout** — Implement 10% → 50% → 100% traffic increase
5. **Safety guardrail** — Add rollback to A/B if bandit performance drops

In [ ]:
# Your experiments here

# Example: Add rollback guardrail
def bandit_with_rollback(threshold_regret=50):
    """Rollback to A/B test if cumulative regret exceeds threshold."""
    # Your implementation
    pass

## Summary

**What you learned:**
- Pure A/B tests accumulate linear regret (keep testing bad variants)
- Pure bandits can have high variance early (insufficient exploration)
- Hybrid approach (A/B burn-in → bandit) balances both concerns
- Optimal burn-in period depends on: arm differences, noise level, risk tolerance

**Migration best practices:**
1. **Start with shadow mode** — Run bandit offline first to validate
2. **Use burn-in period** — Initial A/B test provides reliable starting point
3. **Keep baseline running** — 10% A/B traffic for comparison
4. **Log propensities** — Enable offline evaluation of new policies
5. **Monitor intensively** — First 2-4 weeks require close attention

**Key insight:** You don't have to choose between A/B testing and bandits. Hybrid approaches give you the statistical rigor of A/B tests during burn-in and the efficiency of bandits afterward.

**Next:** Notebook 03 - Deploy complete commodity allocation system with all production components

In [ ]:
key_takeaways(["Start with shadow mode", "Use burn-in period", "Keep baseline running", "Log propensities", "Monitor intensively"])